In [13]:
# Import libraries
import pandas as pd
from pathlib import Path
import numpy as np
import tempfile
import torch
from dataset import StainBagCasePooledFeatureDataset
from attention_analysis import compute_effective_patch_attention

### Check 1: Creating Artificial Patches

#### dataset.py shuffling

In [6]:
# Create 20 artificial patches and corresponding embeddings
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    paths = []
    for i in range(20):
        filename = f"case_1_match_1_h&e_patch{i}.png"
        path = root / filename

        path.touch()
        torch.save(torch.full((4096,), float(i)),
                   root / path.with_suffix(".pt").name)
        paths.append(str(path))
    # Load patches with shuffling enabled
    ds = StainBagCasePooledFeatureDataset(case_dict={1: {"h&e": [paths]}},
                                          label_map={1: 0},
                                          embeddings_dir=str(root),
                                          stains=("h&e",),
                                          per_slice_cap=20,
                                          shuffle_patches=True)

# Trigger 10 shuffles
    for _ in range(10):
        sample = ds[0]
        features = sample["stain_slices"]["h&e"][0]
        filenames = sample["patch_filenames"]["h&e"][0]
        assert len(features) == len(filenames)
        for feature, filename in zip(features, filenames):
            patch_num = int(filename.split("_patch")[-1].split(".")[0])
            # Check
            assert torch.all(feature == float(patch_num)), f"Mismatch for {filename}"

# Print success
print("PASS: Filenames stay aligned with embeddings after shuffling!")

PASS: Filenames stay aligned with embeddings after shuffling!


In [12]:
# Create 20 artificial patches and corresponding embeddings
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    paths = []
    for i in range(20):
        filename = f"case_1_match_1_h&e_patch{i}.png"
        path = root / filename
        path.touch()
        torch.save(torch.full((4096,), float(i)),
                   path.with_suffix(".pt"))
        paths.append(str(path))

    # Load patches with shuffling enabled
    ds = StainBagCasePooledFeatureDataset(
        case_dict={1: {"h&e": [paths]}},
        label_map={1: 0},
        embeddings_dir=str(root),
        stains=("h&e",),
        per_slice_cap=20,
        shuffle_patches=True
    )

    # Track the different patch orders
    orders = set()

    # Trigger 10 shuffles
    for _ in range(10):
        sample = ds[0]
        features = sample["stain_slices"]["h&e"][0]
        filenames = sample["patch_filenames"]["h&e"][0]

        # Check every embedding has a filename
        assert len(features) == len(filenames) == 20

        orders.add(tuple(filenames))

        # Check every embedding matches its filename
        for feature, filename in zip(features, filenames):
            patch_num = int(filename.split("_patch")[-1].split(".")[0])

            assert torch.all(feature == float(patch_num)), f"Mismatch for {filename}"

    # Check shuffling actually changed the patch order
    assert len(orders) > 1, "Shuffling did not change patch order!"

print('Check passed')

Check passed


### Check 2: Looking At Top Effective Patches

#### attention_analysis.py

In [14]:
# Create artificial filenames and weights
filenames = ["patch10.png",
             "patch2.png",
             "patch0.png"]
patch_weights = torch.tensor([0.6, 0.3, 0.1])

# Simulate the outputs
patch_filenames = {"h&e": [filenames]}
attention_weights = {
    "case_weights": torch.tensor([1.0]),
    "stain_order": ["h&e"],
    "stain_weights": {
        "h&e": {
            "slice_weights": torch.tensor([1.0]),
            "patch_weights": [patch_weights]
        }
    }
}

# Run attention analysis
records = compute_effective_patch_attention(
    case_id=1,
    attention_weights=attention_weights,
    patch_filenames=patch_filenames
)

# Check
for i, record in enumerate(records):
    assert record["patch_idx"] == i
    assert record["patch_filename"] == filenames[i]
    assert abs(
        record["patch_attn_weight"] - patch_weights[i].item()
    ) < 1e-6

print("Check passed")

Check passed
